# Bước 3. Tính Attainable Potential bằng cách so với các shop giống nó

Notebook này tính Attainable Potential (mức đạt được nếu can thiệp hợp lý) của từng shop: nếu shop làm tốt bằng 1/4 số shop cùng loại đang làm, shop sẽ ký được bao nhiêu contract mỗi tháng.

**Cách chạy:** chạy Bước 1 và Bước 2 trước. Notebook này đọc `Bước 1/shop_current.csv` (Current Actual và KPI quý 3/2026) và `Bước 2/shop_bau.csv` (BAU 10/2026–03/2027).

**Cách làm, tóm tắt:**
- Chia shop thành peer group (nhóm các shop giống nhau): cùng cỡ shop, cùng ngành hàng chính, cùng kiểu làm hồ sơ, cùng tỉnh.
- Trong mỗi peer group, lấy benchmark (mức chuẩn) là mức mà top 25% shop (1/4 số shop đứng đầu) đã đạt được, cho từng KPI.
- KPI nào của shop kém hơn benchmark thì đưa về bằng benchmark, rồi tính lại số contract.
- Áp hai giới hạn: không giảm tỷ lệ bị từ chối nếu khách của shop trễ hạn nhiều hơn mức chung của nhóm; và kết quả không vượt quá số contract mà 1/4 shop tốt nhất của nhóm đang ký.

**File kết quả:**
- `output/shop_attainable.csv`: mỗi dòng một shop Active, gồm benchmark, KPI sau khi nâng, Attainable Potential, phần tăng đến từ KPI nào, và nhãn độ tin cậy.
- `Bước 3/shop_kiem_chung_ty_le.csv`: kết quả kiểm chứng ở mục 3.10 (chỉ nâng tỷ lệ, giữ nguyên số hồ sơ) cho các shop đủ dữ liệu.

Ô dưới đây đọc hai file kết quả và giữ các shop được đánh giá: shop Active, không phải shop online. Trong 80 shop online đang Active, chỉ 3 shop có hồ sơ trong quý 3/2026 (dòng đầu của kết quả bên dưới), quá ít để so sánh, nên không tính Attainable Potential cho nhóm này.

Trong các shop được đánh giá, **shop đang bán** là shop có ít nhất một hồ sơ trong quý 3/2026. Chỉ các shop đang bán được dùng để tính benchmark, vì shop không có hồ sơ thì không đo được số SA, năng suất hay tỷ lệ ký.

In [1]:
import sys
from pathlib import Path
import numpy as np
ROOT = next(q for q in [Path.cwd(), *Path.cwd().parents] if (q / 'Bước 1').exists())   # thư mục gốc của bài
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from r3_db import connect, save_csv

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100); 
def _fmt(x):
    if x != x: return 'NaN'
    if x == int(x): return f'{x:,.0f}'
    return f'{x:,.1f}' if abs(x) >= 100 else f'{x:,.3f}'
pd.set_option('display.float_format', _fmt)


# Kiểu biểu đồ dùng chung: một màu chính, không có lưới.
BLUE, ORANGE, INK, MUTED = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e'
plt.rcParams.update({'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'axes.edgecolor': '#d9d8d4',
                     'axes.grid': False, 'grid.color': '#ecebe8', 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.labelcolor': MUTED, 'xtick.color': MUTED, 'ytick.color': MUTED, 'font.size': 10,
                     'lines.linewidth': 2})

cur = pd.read_csv(ROOT / 'Bước 1' / 'shop_current.csv', encoding='utf-8-sig')
# BAU của Bước 2: phần bán tại shop, phần cross-sell ký ở nơi khác, và tổng của hai phần
bau = pd.read_csv(ROOT / 'Bước 2' / 'shop_bau.csv', encoding='utf-8-sig', dtype={'shop_id': str})
bau = bau[bau.shop_id != 'VIRTUAL'].assign(skp_salesroom=lambda b: b.shop_id.astype(int)).rename(columns={
    'bau_contracts': 'bau_hd_truc_tiep', 'bau_xsell_elsewhere_contracts': 'bau_xsell', 'bau_total_contribution_contracts': 'bau_hd_tong'})
bau = bau[['skp_salesroom', 'bau_hd_truc_tiep', 'bau_xsell', 'bau_hd_tong']]
d = cur.merge(bau, on='skp_salesroom')
d = d[(d.salesroom_status == 'Active') & (d.shop_online == 0)].copy()
d['dang_ban'] = d.ca_ho_so > 0
dang_ban = d[d.dang_ban]
onl = cur[(cur.salesroom_status == 'Active') & (cur.shop_online == 1)]
print(f'Shop online đang Active: {len(onl):,}; trong đó có hồ sơ trong quý 3/2026: {int((onl.ca_ho_so > 0).sum()):,}')
print(f'Shop được đánh giá: {len(d):,}; trong đó shop đang bán: {len(dang_ban):,}')

Shop online đang Active: 80; trong đó có hồ sơ trong quý 3/2026: 3
Shop được đánh giá: 11,752; trong đó shop đang bán: 8,091


## 3.1. Peer group (nhóm các shop giống nhau)

Một peer group gồm các shop giống nhau ở 4 điểm:
- cỡ shop (`salesroom_category`): Big, Medium, Small;
- ngành hàng chính (`salesroom_product`): CD, TW, NS;
- kiểu làm hồ sơ (`salesroom_model`): Aldi hoặc Non-Aldi;
- tỉnh theo địa giới mới (`salesroom_province`, 34 tỉnh, thành).

Câu lệnh dưới đây đếm số shop đang bán trong mỗi peer group, rồi tóm tắt: có bao nhiêu nhóm, nhóm điển hình có bao nhiêu shop, và bao nhiêu nhóm (bao nhiêu shop) rơi vào nhóm dưới 5 shop và dưới 20 shop.

In [2]:
G = ['salesroom_category', 'salesroom_product', 'salesroom_model', 'salesroom_province']   # peer group
G_QG = G[:3]                                                                              # cùng loại, toàn quốc
co_nhom = dang_ban.groupby(G).size().rename('so_shop')
pd.DataFrame([{'so_nhom': len(co_nhom), 'trung_vi_so_shop': co_nhom.median(), 'nhom_lon_nhat': co_nhom.max(),
  'nhom_duoi_5_shop': int((co_nhom < 5).sum()), 'so_shop_trong_nhom_duoi_5': int(co_nhom[co_nhom < 5].sum()),
  'nhom_duoi_20_shop': int((co_nhom < 20).sum()), 'so_shop_trong_nhom_duoi_20': int(co_nhom[co_nhom < 20].sum())}])

,so_nhom,trung_vi_so_shop,nhom_lon_nhat,nhom_duoi_5_shop,so_shop_trong_nhom_duoi_5,nhom_duoi_20_shop,so_shop_trong_nhom_duoi_20
0,313,11,328,91,192,204,1374


- Có 313 peer group có shop đang bán. Nhóm điển hình chỉ có 11 shop; nhóm lớn nhất có 328 shop.
- 91 nhóm có dưới 5 shop, chứa 192 shop. Với các nhóm này, mức 75% tính trên 1–4 shop không có ý nghĩa, nên benchmark được lấy từ cấp toàn quốc (mục 3.2).
- 204 nhóm có dưới 20 shop, chứa 1.374 trên 8.091 shop đang bán (17%). Kết quả của các shop này được gắn nhãn độ tin cậy thấp (mục 3.8).

## 3.2. Benchmark của từng KPI

Doanh số bán trực tiếp của shop tách được thành các KPI sau (Bước 1, mục 1.7):

**số contract mỗi tháng = số SA × số hồ sơ mỗi SA × (1 − tỷ lệ bị từ chối − tỷ lệ hủy)**

Phần cross-sell được tính bằng: **số lead mỗi tháng × số contract cross-sell trên mỗi lead**. Số lead do công ty giao nên giữ nguyên, chỉ xét nâng số contract trên mỗi lead.

Benchmark của một peer group là mức mà 1/4 số shop tốt nhất trong nhóm đã đạt được:
- với số SA, hồ sơ mỗi SA và cross-sell mỗi lead (càng cao càng tốt): **mức 75%**, tức xếp các shop từ thấp đến cao, lấy mức mà 1/4 số shop đứng đầu đã đạt hoặc vượt;
- với tỷ lệ bị từ chối và tỷ lệ hủy (càng thấp càng tốt): **mức 25%**, tức mức mà 1/4 số shop có tỷ lệ thấp nhất đã đạt.

Ba KPI cần thêm điều kiện để con số không bị méo:
- **tỷ lệ bị từ chối và tỷ lệ hủy:** chỉ tính benchmark trên các shop có từ 10 hồ sơ trở lên trong quý 3/2026. Với shop chỉ có 1–2 hồ sơ, các tỷ lệ này chỉ có thể là 0%, 50% hoặc 100%, nên nếu tính cả các shop này thì benchmark thường sẽ là 0%;
- **contract cross-sell trên mỗi lead:** chỉ tính benchmark trên các shop có từ 30 lead trở lên trong quý.

Ngoài ra lấy thêm mức 75% của chính số contract bán trực tiếp mỗi tháng, dùng làm giới hạn ở mục 3.5.

Nếu trong peer group có dưới 5 shop đủ điều kiện, benchmark được lấy từ các shop cùng loại trên toàn quốc (cùng cỡ shop, ngành hàng, kiểu làm hồ sơ, bỏ điều kiện cùng tỉnh). Cột `..._nguon` ghi benchmark lấy từ cấp nào.

Ô dưới đây định nghĩa hàm tính benchmark và áp cho cả 6 cột.

In [3]:
SO_SHOP_TOI_THIEU = 5
def benchmark(cot, nen, ten, muc=0.75):
    tinh = nen.groupby(G)[cot].agg(n_tinh='count', bm_tinh=lambda s: s.quantile(muc)).reset_index()
    qg = nen.groupby(G_QG)[cot].agg(n_qg='count', bm_qg=lambda s: s.quantile(muc)).reset_index()
    o = d[['skp_salesroom'] + G].merge(tinh, on=G, how='left').merge(qg, on=G_QG, how='left')
    dung_tinh = o.n_tinh.fillna(0) >= SO_SHOP_TOI_THIEU
    o[ten] = np.where(dung_tinh, o.bm_tinh, o.bm_qg)
    o[ten + '_nguon'] = np.where(dung_tinh, 'peer group', 'toàn quốc')
    return o[['skp_salesroom', ten, ten + '_nguon']]

du_10_hs = dang_ban[dang_ban.ca_ho_so * 3 >= 10]
CAC_KPI = [('ca_so_sa', dang_ban, 'bm_so_sa', 0.75),
           ('ca_ho_so_moi_sa', dang_ban, 'bm_ho_so_moi_sa', 0.75),
           ('ca_ty_le_tu_choi', du_10_hs, 'bm_ty_le_tu_choi', 0.25),
           ('ca_ty_le_huy', du_10_hs, 'bm_ty_le_huy', 0.25),
           ('ca_xsell_moi_lead', dang_ban[dang_ban.ca_lead * 3 >= 30], 'bm_xsell_moi_lead', 0.75),
           ('ca_hd_truc_tiep', dang_ban, 'bm_hd_truc_tiep', 0.75)]
x = d.copy()
for cot, nen, ten, muc in CAC_KPI:
    x = x.merge(benchmark(cot, nen, ten, muc), on='skp_salesroom')
xb = x[x.dang_ban]
pd.DataFrame({ten: xb[ten + '_nguon'].value_counts() for _, _, ten, _ in CAC_KPI}).T

,peer group,toàn quốc
bm_so_sa,7899,192
bm_ho_so_moi_sa,7899,192
bm_ty_le_tu_choi,7226,865
bm_ty_le_huy,7226,865
bm_xsell_moi_lead,5549,2542
bm_hd_truc_tiep,7899,192


Phần lớn shop dùng được benchmark của chính peer group. Riêng tỷ lệ bị từ chối và tỷ lệ hủy (865 shop) và cross-sell mỗi lead (2.542 shop) phải dùng benchmark toàn quốc nhiều hơn, vì điều kiện tối thiểu về số hồ sơ và số lead.

Để dễ đọc, câu lệnh dưới đây in benchmark ở cấp toàn quốc của từng loại shop (cùng cỡ, ngành hàng, kiểu làm hồ sơ), kèm số shop đang bán và Current Actual trung vị của loại đó. Benchmark thật của mỗi shop là benchmark trong peer group của nó, có thể cao hoặc thấp hơn.

In [4]:
bang_qg = dang_ban.groupby(G_QG).agg(
    so_shop=('skp_salesroom', 'size'),
    hd_trung_vi=('ca_hd_truc_tiep', 'median'),
    bm_so_sa=('ca_so_sa', lambda s: s.quantile(0.75)),
    bm_ho_so_moi_sa=('ca_ho_so_moi_sa', lambda s: s.quantile(0.75)),
    bm_hd_truc_tiep=('ca_hd_truc_tiep', lambda s: s.quantile(0.75)))
bang_qg['bm_ty_le_tu_choi'] = du_10_hs.groupby(G_QG).ca_ty_le_tu_choi.quantile(0.25)
bang_qg['bm_ty_le_huy'] = du_10_hs.groupby(G_QG).ca_ty_le_huy.quantile(0.25)
bang_qg['bm_xsell_moi_lead'] = dang_ban[dang_ban.ca_lead * 3 >= 30].groupby(G_QG).ca_xsell_moi_lead.quantile(0.75)
bang_qg

so_shop  hd_trung_vi  bm_so_sa  bm_ho_so_moi_sa  bm_hd_truc_tiep  bm_ty_le_tu_choi  bm_ty_le_huy  bm_xsell_moi_lead
salesroom_category salesroom_product salesroom_model                                                                                                                     
Big                CD                Aldi                  16       79.333    14.333           18.195            159.8             0.195         0.085              0.006
                                     Non-Aldi             324            2     1.667                6                7             0.100         0.029              0.054
                   NS                Aldi                 268        2.167     1.667            5.186            6.417             0.118         0.036              0.031
                   TW                Non-Aldi             288        3.333     1.667            5.714            7.667                 0         0.037              0.068
Medium             CD                Aldi                  15       27.333         7               18           45.667             0.226         0.067              0.001
                                     Non-Aldi            1315            2     1.667                5            5.667             0.116         0.016              0.057
                   NS                Aldi                1005        1.667     1.667            4.500            4.667             0.107         0.029              0.016
                   TW                Non-Aldi             651            2     1.667            4.400            5.500                 0         0.020              0.083
Small              CD                Aldi                   9           45     9.333           13.933           74.333             0.191         0.068              0.006
                                     Non-Aldi            2711        1.333     1.333            4.050                4             0.108             0              0.050
                   NS                Aldi                 829        1.667         1            4.750                4             0.150             0              0.018
                   TW                Non-Aldi             660        1.667     1.333            4.025            4.333                 0         0.022              0.062

- Với các loại shop phổ biến (CD Non-Aldi, NS Aldi, TW Non-Aldi), 1/4 shop tốt nhất ký 4–7,7 contract mỗi tháng, trong khi shop đứng giữa chỉ ký 1,3–3,3. Khoảng cách giữa shop điển hình và shop tốt trong cùng loại là rất lớn.
- Ở 1/4 shop tốt nhất, shop TW gần như không có hồ sơ bị từ chối (benchmark 0%), trong khi shop CD Non-Aldi và NS vẫn bị từ chối 10–15% hồ sơ. Tỷ lệ hủy của 1/4 shop tốt nhất chỉ 0–4% ở shop CD Non-Aldi, NS và TW, nhưng 7–9% ở shop CD Aldi.
- Shop CD Aldi khác hẳn: chỉ có 40 shop đang bán (16 + 15 + 9), nhưng shop đứng giữa ký 27–79 contract mỗi tháng và benchmark có 7–14 SA. Đây là các cửa hàng lớn mà nhân viên của chính cửa hàng làm hồ sơ. Nếu không tách theo kiểu làm hồ sơ, các shop này sẽ kéo benchmark của shop CD lên rất cao.

## 3.3. Shop thấp hơn benchmark ở KPI nào

Câu lệnh dưới đây đếm, trong các shop đang bán, bao nhiêu shop kém hơn benchmark của peer group ở từng KPI (thấp hơn với số SA, hồ sơ mỗi SA, cross-sell mỗi lead; cao hơn với tỷ lệ bị từ chối và tỷ lệ hủy), và mức chênh trung vị ở những shop đó. Với ba KPI đầu, mức chênh là benchmark chia KPI của shop. Với hai tỷ lệ, mức chênh là KPI của shop trừ benchmark, tính bằng điểm phần trăm, vì benchmark có thể bằng 0.

In [5]:
rows = []
for cot, ten, nhan, cao_la_tot in [('ca_so_sa', 'bm_so_sa', 'số SA', True), ('ca_ho_so_moi_sa', 'bm_ho_so_moi_sa', 'hồ sơ mỗi SA', True),
                       ('ca_xsell_moi_lead', 'bm_xsell_moi_lead', 'cross-sell mỗi lead', True),
                       ('ca_ty_le_tu_choi', 'bm_ty_le_tu_choi', 'tỷ lệ bị từ chối', False), ('ca_ty_le_huy', 'bm_ty_le_huy', 'tỷ lệ hủy', False)]:
    co = xb[cot].notna() & xb[ten].notna()
    kem = co & ((xb[cot] < xb[ten]) if cao_la_tot else (xb[cot] > xb[ten]))
    rows.append({'kpi': nhan, 'so_shop_co_so_lieu': int(co.sum()), 'so_shop_kem_hon': int(kem.sum()),
                 'ty_le_kem_hon': kem.sum() / co.sum(),
                 'benchmark_chia_kpi_trung_vi': (xb.loc[kem, ten] / xb.loc[kem, cot].replace(0, np.nan)).median() if cao_la_tot else np.nan,
                 'chenh_diem_phan_tram_trung_vi': np.nan if cao_la_tot else ((xb.loc[kem, cot] - xb.loc[kem, ten]) * 100).median()})
pd.DataFrame(rows)

,kpi,so_shop_co_so_lieu,so_shop_kem_hon,ty_le_kem_hon,benchmark_chia_kpi_trung_vi,chenh_diem_phan_tram_trung_vi
0,số SA,8091,5237,0.647,2,NaN
1,hồ sơ mỗi SA,8091,5934,0.733,2.375,NaN
2,cross-sell mỗi lead,7335,5681,0.775,1.667,NaN
3,tỷ lệ bị từ chối,8091,4303,0.532,NaN,15
4,tỷ lệ hủy,8091,3502,0.433,NaN,9.189


- 65% shop đang bán có số SA thấp hơn benchmark; ở các shop này, benchmark trung vị gấp 2 lần số SA hiện có, ví dụ shop đang có 1 SA thì benchmark là 2 SA.
- 73% shop có số hồ sơ mỗi SA thấp hơn benchmark, và benchmark trung vị gấp 2,4 lần mức của shop.
- 77% shop có cross-sell mỗi lead thấp hơn benchmark, benchmark trung vị gấp 1,7 lần.
- 53% shop có tỷ lệ bị từ chối cao hơn benchmark; ở shop đứng giữa trong số này, tỷ lệ cao hơn benchmark 15 điểm phần trăm.
- 43% shop có tỷ lệ hủy cao hơn benchmark, cao hơn 9,2 điểm phần trăm ở shop đứng giữa.

Như vậy khoảng cách lớn nhất nằm ở số hồ sơ shop tạo ra (số SA và hồ sơ mỗi SA), không phải ở tỷ lệ bị từ chối hay tỷ lệ hủy. Số SA và số hồ sơ mỗi SA thấp hơn benchmark khoảng 2 lần, còn hai tỷ lệ chỉ chênh 9–15 điểm phần trăm.

## 3.4. Giới hạn vì risk (rủi ro)

Giảm tỷ lệ bị từ chối nghĩa là duyệt thêm hồ sơ mà trước đây công ty từ chối. Nếu khách của shop đã trả nợ kém hơn mức chung, duyệt thêm có thể làm công ty mất tiền. Vì vậy tỷ lệ bị từ chối của shop chỉ được giảm khi có đủ hai điều kiện:
- shop có từ 30 contract trở lên trong nhóm tính rủi ro (ký từ 03/2025 đến hết 05/2026). Bước 1 cho thấy với ít contract hơn, tỷ lệ trễ hạn của shop chưa đủ tin cậy;
- tỷ lệ contract có khách trễ hạn trong 4 kỳ đầu của shop không cao hơn tỷ lệ chung của peer group (cộng tất cả contract tính rủi ro của các shop trong nhóm).

Các KPI còn lại không bị giới hạn này: số SA, hồ sơ mỗi SA và cross-sell làm tăng số hồ sơ; giảm tỷ lệ hủy là giữ lại những khách đã đến shop mà không phải nới điều kiện duyệt. Riêng tỷ lệ hủy chỉ được giảm ở shop có từ 10 hồ sơ trong quý, vì tỷ lệ của shop ít hồ sơ chưa đủ tin cậy (mục 3.2).

Câu lệnh dưới đây tính tỷ lệ trễ hạn chung của từng peer group, rồi đếm số shop đang bán được phép giảm tỷ lệ bị từ chối.

In [6]:
rr = d.assign(tre=d.rr_hd * d.rr_ty_le_tre.fillna(0)).groupby(G).agg(tre=('tre', 'sum'), hd=('rr_hd', 'sum'))
rr['rr_peer_group'] = rr.tre / rr.hd.replace(0, np.nan)
x = x.merge(rr[['rr_peer_group']].reset_index(), on=G, how='left')
x['du_30_hd_rui_ro'] = x.rr_hd >= 30
x['duoc_giam_tu_choi'] = x.du_30_hd_rui_ro & (x.rr_ty_le_tre <= x.rr_peer_group)
xb = x[x.dang_ban]
pd.DataFrame([{'shop_dang_ban': len(xb), 'du_30_hd_rui_ro': int(xb.du_30_hd_rui_ro.sum()),
               'tre_han_khong_cao_hon_nhom': int(xb.duoc_giam_tu_choi.sum()),
               'duoc_giam_tu_choi_va_dang_cao_hon_bm': int((xb.duoc_giam_tu_choi & (xb.ca_ty_le_tu_choi > xb.bm_ty_le_tu_choi)).sum()),
               'du_10_ho_so_va_huy_cao_hon_bm': int(((xb.ca_ho_so * 3 >= 10) & (xb.ca_ty_le_huy > xb.bm_ty_le_huy)).sum())}])

,shop_dang_ban,du_30_hd_rui_ro,tre_han_khong_cao_hon_nhom,duoc_giam_tu_choi_va_dang_cao_hon_bm,du_10_ho_so_va_huy_cao_hon_bm
0,8091,1733,934,653,2412


- Chỉ 1.733 trên 8.091 shop đang bán có đủ 30 contract tính rủi ro.
- Trong số đó, 934 shop có tỷ lệ trễ hạn không cao hơn mức chung của nhóm.
- Cuối cùng chỉ 653 shop vừa được phép giảm tỷ lệ bị từ chối, vừa đang có tỷ lệ bị từ chối cao hơn benchmark. So với 4.303 shop có tỷ lệ bị từ chối cao hơn benchmark (mục 3.3), phần lớn shop không được giảm: hoặc vì chưa đủ contract để biết rủi ro, hoặc vì khách đang trễ hạn nhiều hơn mức chung.
- Tỷ lệ hủy không bị giới hạn này: 2.412 shop có từ 10 hồ sơ trong quý và có tỷ lệ hủy cao hơn benchmark sẽ được giảm.

## 3.5. Tính lại doanh số và giới hạn theo shop tốt nhất của nhóm

Ô dưới đây tính cho mỗi shop đang bán:
- `sau_so_sa`, `sau_ho_so_moi_sa`: lấy giá trị lớn hơn giữa KPI của shop và benchmark;
- `sau_ty_le_tu_choi`: giá trị nhỏ hơn giữa tỷ lệ của shop và benchmark, nếu shop được phép giảm (mục 3.4); nếu không thì giữ nguyên;
- `sau_ty_le_huy`: giá trị nhỏ hơn giữa tỷ lệ của shop và benchmark, nếu shop có từ 10 hồ sơ trong quý; nếu không thì giữ nguyên;
- `sau_ty_le_ky` = 1 − `sau_ty_le_tu_choi` − `sau_ty_le_huy`;
- `tt_tinh_lai` = `sau_so_sa` × `sau_ho_so_moi_sa` × `sau_ty_le_ky`: số contract bán trực tiếp sau khi tính lại.

Đưa cùng lúc tất cả KPI về benchmark có thể cho ra một con số mà rất ít shop trong nhóm thực sự đạt được, vì một shop đứng ở nhóm 1/4 tốt nhất ở mọi KPI hiếm hơn nhiều so với đứng ở nhóm tốt nhất ở một KPI. Ô dưới đây kiểm tra điều này: đếm bao nhiêu shop có `tt_tinh_lai` vượt mức 75% của chính số contract bán trực tiếp trong peer group (`bm_hd_truc_tiep`).

In [7]:
db = x.dang_ban
x['sau_so_sa'] = np.where(db, np.maximum(x.ca_so_sa, x.bm_so_sa), np.nan)
x['sau_ho_so_moi_sa'] = np.where(db, np.maximum(x.ca_ho_so_moi_sa, x.bm_ho_so_moi_sa), np.nan)
x['sau_ty_le_tu_choi'] = np.where(db, np.where(x.duoc_giam_tu_choi, np.minimum(x.ca_ty_le_tu_choi, x.bm_ty_le_tu_choi.fillna(1)), x.ca_ty_le_tu_choi), np.nan)
x['sau_ty_le_huy'] = np.where(db, np.where(x.ca_ho_so * 3 >= 10, np.minimum(x.ca_ty_le_huy, x.bm_ty_le_huy.fillna(1)), x.ca_ty_le_huy), np.nan)
x['sau_ty_le_ky'] = 1 - x.sau_ty_le_tu_choi - x.sau_ty_le_huy
x['tt_tinh_lai'] = np.where(db, x.sau_so_sa * x.sau_ho_so_moi_sa * x.sau_ty_le_ky, 0)
x['tran_nhom'] = np.maximum(x.bm_hd_truc_tiep, x.ca_hd_truc_tiep)   # shop đã vượt mức 75% thì giữ mức của chính shop
vuot = db & (x.tt_tinh_lai > x.tran_nhom)
kq = pd.DataFrame([{'shop_dang_ban': int(db.sum()), 'so_shop_vuot_muc_75_cua_nhom': int(vuot.sum()), 'ty_le': vuot.sum() / db.sum(),
               'tong_hd_hien_tai': x.loc[db, 'ca_hd_truc_tiep'].sum(), 'tong_tt_tinh_lai': x.loc[db, 'tt_tinh_lai'].sum(),
               'tong_neu_gioi_han': np.minimum(x.tt_tinh_lai, x.tran_nhom)[db].sum()}])
kq['tang_neu_khong_gioi_han'] = kq.tong_tt_tinh_lai / kq.tong_hd_hien_tai - 1
kq['tang_neu_gioi_han'] = kq.tong_neu_gioi_han / kq.tong_hd_hien_tai - 1
kq

,shop_dang_ban,so_shop_vuot_muc_75_cua_nhom,ty_le,tong_hd_hien_tai,tong_tt_tinh_lai,tong_neu_gioi_han,tang_neu_khong_gioi_han,tang_neu_gioi_han
0,8091,5920,0.732,"37,664","70,958.7","55,092.9",0.884,0.463


- Nếu đưa mọi KPI về benchmark, tổng contract bán trực tiếp của các shop đang bán tăng từ 37.664 lên 70.959 mỗi tháng, tức tăng 88% (cột `tang_neu_khong_gioi_han`).
- Nhưng 5.920 shop (73%) khi đó sẽ vượt mức 75% số contract của chính peer group, tức vượt mức mà 1/4 shop tốt nhất cùng loại đang thực sự ký. Con số đó không còn là mức "nhiều shop cùng loại đã làm được".
- Vì vậy kết quả được giới hạn ở `tran_nhom`: không vượt mức 75% số contract của peer group (shop nào đã vượt mức này thì giữ mức của chính shop). Sau giới hạn, tổng là 55.093 contract mỗi tháng, tăng 46% so với hiện tại (cột `tang_neu_gioi_han`).

## 3.6. Attainable Potential

Ô dưới đây tính:
- `ap_truc_tiep` = `tt_tinh_lai`, nhưng không vượt `tran_nhom`;
- `ap_xsell` = số lead mỗi tháng × số contract cross-sell trên mỗi lead sau khi nâng lên benchmark, nhưng không thấp hơn cross-sell hiện tại. Một shop có thể có cross-sell trong quý 3/2026 mà không có lead tạo trong quý, vì contract ký trong quý có thể đến từ lead tạo trước đó (trong vòng 30 ngày);
- `ap_hd_tong` = `ap_truc_tiep` + `ap_xsell`, nhưng không thấp hơn BAU (`bau_hd_tong`). Lý do: BAU là mức shop đạt được khi không làm gì; nếu xu hướng của shop (ví dụ shop mới đang tăng) đã đưa shop lên cao hơn mức tính lại, thì Attainable Potential ít nhất phải bằng BAU.

Shop được đánh giá nhưng không có hồ sơ nào trong quý 3/2026 không đo được KPI nên không tính lại được. Với các shop này, Attainable Potential được để bằng BAU và gắn nhãn `khong_hoat_dong_q3`.

In [8]:
x['ap_truc_tiep'] = np.where(db, np.minimum(x.tt_tinh_lai, x.tran_nhom), 0)
x['sau_xsell_moi_lead'] = np.where(db, np.maximum(x.ca_xsell_moi_lead.fillna(0), x.bm_xsell_moi_lead.fillna(0)), np.nan)
x['ap_xsell'] = np.where(db, np.maximum(x.ca_lead * x.sau_xsell_moi_lead, x.ca_xsell), x.ca_xsell)
x['ap_truoc_bau'] = x.ap_truc_tiep + x.ap_xsell
x['ap_hd_tong'] = np.maximum(x.ap_truoc_bau, x.bau_hd_tong)
x['khong_hoat_dong_q3'] = (~db).astype(int)
x['ap_bang_bau'] = (x.bau_hd_tong >= x.ap_truoc_bau).astype(int)
pd.DataFrame({'Current Actual': [x.ca_hd_truc_tiep.sum(), x.ca_xsell.sum(), x.ca_hd_tong.sum()],
              'BAU': [x.bau_hd_truc_tiep.sum(), x.bau_xsell.sum(), x.bau_hd_tong.sum()],
              'Attainable Potential': [x.ap_truc_tiep.sum(), x.ap_xsell.sum(), x.ap_hd_tong.sum()]},
             index=['bán trực tiếp', 'cross-sell', 'tổng (không thấp hơn BAU)']).T

,bán trực tiếp,cross-sell,tổng (không thấp hơn BAU)
Current Actual,"37,664","1,786","39,450"
BAU,"37,733.6","1,538.0","39,271.6"
Attainable Potential,"55,092.9","3,229.1","59,861.1"


- Cross-sell tăng nhiều nhất về tỷ lệ: Attainable Potential gấp khoảng 1,8 lần mức hiện tại (3.229 / 1.786 ≈ 1,8), vì hơn 3/4 số shop đang có cross-sell mỗi lead thấp hơn benchmark (mục 3.3).
- Tổng Attainable Potential là 59.861 contract mỗi tháng, cao hơn BAU (39.272) khoảng 52% (59.861 / 39.272 ≈ 1,52).

Câu lệnh dưới đây so Attainable Potential với BAU theo ngành hàng và cỡ shop, chỉ tính các shop đang bán: tổng BAU, tổng Attainable Potential, mức cao hơn BAU (phần trăm), và trung vị theo từng shop.

In [9]:
y = x[x.dang_ban]
t = y.groupby(['salesroom_product', 'salesroom_category']).agg(so_shop=('skp_salesroom', 'size'), bau=('bau_hd_tong', 'sum'),
      ap=('ap_hd_tong', 'sum'), bau_trung_vi=('bau_hd_tong', 'median'), ap_trung_vi=('ap_hd_tong', 'median'))
t['ap_cao_hon_bau'] = t.ap / t.bau - 1
t

so_shop      bau       ap  bau_trung_vi  ap_trung_vi  ap_cao_hon_bau
salesroom_product salesroom_category                                                                      
CD                Big                     340  4,177.4  6,429.5         3.125        7.313           0.539
                  Medium                 1330  7,365.5 11,020.1         2.743        6.169           0.496
                  Small                  2720 10,655.4 15,679.3         1.897            4           0.471
NS                Big                     268  1,467.3  2,341.6         2.344        6.859           0.596
                  Medium                 1005  4,228.9  6,524.3         1.667        5.381           0.543
                  Small                   829  3,299.7  4,854.2         1.745        4.048           0.471
TW                Big                     288  1,589.1  2,709.2         3.146        8.417           0.705
                  Medium                  651  3,289.3  5,325.7         2.307        7.163           0.619
                  Small                   660  2,693.3  4,449.0         1.890        5.021           0.652

- Shop CD và NS có Attainable Potential cao hơn BAU khoảng 47–60%. Shop TW cao hơn nhiều hơn, khoảng 62–71%.
- Theo từng shop, mức chênh còn lớn hơn: ví dụ shop CD Small đứng giữa có BAU 1,9 contract mỗi tháng nhưng Attainable Potential 4,0, tức gấp khoảng 2,1 lần (4,0 / 1,9), trong khi tổng của cả nhóm CD Small chỉ cao hơn BAU 47,1%. Nghĩa là các shop bán ít được nâng nhiều hơn về tỷ lệ so với các shop bán nhiều.

## 3.7. Phần tăng đến từ KPI nào

Phần chênh giữa Attainable Potential và BAU của mỗi shop được chia thành các phần sau:
- **giữ mức hiện tại** (`tang_do_giu_muc_hien_tai`): nếu BAU thấp hơn Current Actual (shop đang đi xuống), thì một phần chênh lệch chỉ là kéo shop về lại mức hiện tại. Phần này bằng Current Actual trừ BAU, hoặc bằng 0 nếu BAU không thấp hơn Current Actual;
- **các KPI bán trực tiếp** (số SA, hồ sơ mỗi SA, tỷ lệ ký): phần tăng của bán trực tiếp được chia theo logarit của tỷ số "sau / trước". Ví dụ số SA tăng gấp 2 và năng suất tăng gấp 1,5 thì số SA được chia phần lớn hơn (log 2 lớn hơn log 1,5). Cách chia này không phụ thuộc thứ tự nâng KPI nào trước. Phần của tỷ lệ ký được chia tiếp cho tỷ lệ bị từ chối và tỷ lệ hủy theo số điểm phần trăm mỗi tỷ lệ giảm. Ví dụ tỷ lệ bị từ chối giảm 3 điểm và tỷ lệ hủy giảm 1 điểm thì tỷ lệ bị từ chối nhận 3/4 phần này;
- **cross-sell**: chênh lệch giữa `ap_xsell` và `ca_xsell`.

Các phần KPI được nhân cùng một hệ số để tổng tất cả các phần đúng bằng phần chênh giữa Attainable Potential và BAU.

Cột `kpi_chinh` ghi KPI đóng góp nhiều nhất cho từng shop. Câu lệnh dưới đây tổng hợp phần tăng theo KPI và đếm số shop theo `kpi_chinh`.

In [10]:
def log_ty_so(sau, truoc):
    return np.log(np.where((truoc > 0) & (sau > 0), sau / np.where(truoc > 0, truoc, 1), 1))
l_sa = log_ty_so(x.sau_so_sa, x.ca_so_sa); l_ns = log_ty_so(x.sau_ho_so_moi_sa, x.ca_ho_so_moi_sa)
l_ky = log_ty_so(x.sau_ty_le_ky, x.ca_ty_le_ky); tong_log = l_sa + l_ns + l_ky
giam_tc = np.nan_to_num(x.ca_ty_le_tu_choi - x.sau_ty_le_tu_choi); giam_huy = np.nan_to_num(x.ca_ty_le_huy - x.sau_ty_le_huy)
phan_tc = np.where(giam_tc + giam_huy > 0, giam_tc / np.where(giam_tc + giam_huy > 0, giam_tc + giam_huy, 1), 0)
tang_tt = np.where(db, x.ap_truc_tiep - x.ca_hd_truc_tiep, 0)
chia = lambda l: np.where(tong_log > 0, tang_tt * l / np.where(tong_log > 0, tong_log, 1), 0)
p = pd.DataFrame({'tang_do_so_sa': chia(l_sa), 'tang_do_ho_so_moi_sa': chia(l_ns), 'tang_do_ty_le_tu_choi': chia(l_ky) * phan_tc, 'tang_do_ty_le_huy': chia(l_ky) * (1 - phan_tc),
                  'tang_do_xsell': np.where(db, x.ap_xsell - x.ca_xsell, 0)}, index=x.index)
khoang = x.ap_hd_tong - x.bau_hd_tong
giu = np.clip(x.ca_hd_tong - x.bau_hd_tong, 0, None).clip(upper=khoang)
tong_p = p.sum(axis=1)
he = np.where(tong_p > 0, (khoang - giu) / np.where(tong_p > 0, tong_p, 1), 0)
p = p.mul(he, axis=0)
p.insert(0, 'tang_do_giu_muc_hien_tai', giu)
x = pd.concat([x, p], axis=1)
x['kpi_chinh'] = np.where(p.sum(axis=1) > 1e-9, p.idxmax(axis=1).str.replace('tang_do_', ''), 'không có')
print(f"Kiểm tra: tổng các phần = {p.sum().sum():,.1f}; Attainable Potential - BAU = {khoang.sum():,.1f}")
pd.DataFrame({'phan_tang': p.sum(), 'ty_trong': p.sum() / p.sum().sum()}).join(
    x[x.dang_ban].kpi_chinh.value_counts().rename(lambda s: 'tang_do_' + s).rename('so_shop_co_kpi_chinh_nay'))

Kiểm tra: tổng các phần = 20,589.5; Attainable Potential - BAU = 20,589.5


,phan_tang,ty_trong,so_shop_co_kpi_chinh_nay
tang_do_giu_muc_hien_tai,"3,466.3",0.168,1565
tang_do_so_sa,"6,744.1",0.328,1789
tang_do_ho_so_moi_sa,"8,784.3",0.427,3084
tang_do_ty_le_tu_choi,85.738,0.004,18
tang_do_ty_le_huy,339.7,0.016,56
tang_do_xsell,"1,169.3",0.057,493


- **Hồ sơ mỗi SA** là nguồn tăng lớn nhất: 42,7% tổng phần chênh, và là KPI chính của 3.084 shop. Đây là nhóm shop có SA nhưng mỗi SA làm được ít hồ sơ.
- **Số SA** đóng góp 32,8%, là KPI chính của 1.789 shop.
- **Giữ mức hiện tại** đóng góp 16,8%: đây là phần chỉ cần chặn đà giảm mà Bước 2 dự báo. Với 1.565 shop, đây là phần lớn nhất, tức các shop này đã gần benchmark và chủ yếu cần giữ được mức hiện tại.
- **Cross-sell** đóng góp 5,7%, là KPI chính của 493 shop.
- **Tỷ lệ hủy** đóng góp 1,6% (KPI chính của 56 shop) và **tỷ lệ bị từ chối** chỉ 0,4% (18 shop). Tỷ lệ bị từ chối nhỏ vì chỉ 653 shop được phép giảm (mục 3.4). Cả hai đều nhỏ vì mức chênh của hai tỷ lệ nhỏ hơn nhiều so với số SA và hồ sơ mỗi SA (mục 3.3), và 73% shop đã chạm giới hạn của nhóm ở mục 3.5 trước khi phần của hai tỷ lệ được tính hết.

Phần tăng đến từ KPI nào gợi ý loại can thiệp: thiếu SA thì thêm người, năng suất thấp thì đào tạo hoặc giao thêm khách, nhiều hồ sơ bị hủy thì SA theo sát khách đến lúc ký, cross-sell thấp thì cải thiện cách chăm sóc lead.

## 3.8. Nhãn độ tin cậy

Ô dưới đây gắn nhãn `do_tin_cay` cho Attainable Potential của từng shop:
- **thấp** nếu có một trong các điều sau: peer group có dưới 20 shop đang bán; ít nhất một benchmark phải lấy từ toàn quốc; shop mới (dưới 6 tháng);
- **không tính được** nếu shop không có hồ sơ trong quý 3/2026;
- **bình thường** với các shop còn lại.

Bước 7 dùng nhãn này khi đánh giá reliability (độ tin cậy) của kết quả.

In [11]:
x = x.merge(co_nhom.rename('so_shop_peer_group').reset_index(), on=G, how='left')
x['so_shop_peer_group'] = x.so_shop_peer_group.fillna(0).astype(int)
dung_qg = (x[[ten + '_nguon' for _, _, ten, _ in CAC_KPI[:4]]] == 'toàn quốc').any(axis=1)
x['do_tin_cay'] = np.select([~x.dang_ban, (x.so_shop_peer_group < 20) | dung_qg | (x.shop_moi == 1)],
                            ['không tính được', 'thấp'], 'bình thường')
x.do_tin_cay.value_counts().to_frame('so_shop')

,so_shop
do_tin_cay,
bình thường,4748
không tính được,3661
thấp,3343


Có 3.661 trên 11.752 shop được đánh giá (31%) không tính được Attainable Potential vì không có hồ sơ nào trong quý 3/2026. Với các shop này, câu hỏi cho ban quản lý không phải là tăng bao nhiêu mà là có nên tiếp tục duy trì shop hay không.

## 3.9. Ghi kết quả

In [12]:
cot = ['skp_salesroom'] + G + ['shop_moi', 'so_thang', 'dang_ban', 'khong_hoat_dong_q3', 'so_shop_peer_group', 'do_tin_cay',
       'ca_hd_truc_tiep', 'ca_xsell', 'ca_hd_tong', 'bau_hd_truc_tiep', 'bau_xsell', 'bau_hd_tong',
       'ca_so_sa', 'bm_so_sa', 'sau_so_sa', 'ca_ho_so_moi_sa', 'bm_ho_so_moi_sa', 'sau_ho_so_moi_sa',
       'ca_ty_le_ky', 'sau_ty_le_ky', 'ca_ty_le_tu_choi', 'bm_ty_le_tu_choi', 'sau_ty_le_tu_choi', 'ca_ty_le_huy', 'bm_ty_le_huy', 'sau_ty_le_huy',
       'rr_hd', 'rr_ty_le_tre', 'rr_peer_group', 'duoc_giam_tu_choi',
       'ca_lead', 'ca_xsell_moi_lead', 'bm_xsell_moi_lead', 'sau_xsell_moi_lead', 'bm_hd_truc_tiep', 'tt_tinh_lai',
       'ap_truc_tiep', 'ap_xsell', 'ap_hd_tong', 'ap_bang_bau',
       'tang_do_giu_muc_hien_tai', 'tang_do_so_sa', 'tang_do_ho_so_moi_sa', 'tang_do_ty_le_tu_choi', 'tang_do_ty_le_huy', 'tang_do_xsell', 'kpi_chinh'] + \
      [ten + '_nguon' for _, _, ten, _ in CAC_KPI]
save_csv(x[cot].sort_values('skp_salesroom'), 'shop_attainable')

Đã ghi output/shop_attainable.csv: 11,752 dòng


## 3.10. Kiểm chứng: chỉ nâng tỷ lệ, giữ nguyên số hồ sơ

Attainable Potential ở trên nâng cả số hồ sơ (số SA, hồ sơ mỗi SA) lẫn các tỷ lệ. Phần nâng số hồ sơ giả định shop có thêm khách, điều dữ liệu chưa chứng minh được. Mục này đo riêng phần tăng **không cần thêm khách**: giữ nguyên số hồ sơ và số lead shop đang có, chỉ nâng tỷ lệ hồ sơ ký được và số contract cross-sell trên mỗi lead. Cách làm theo bài của Nguyên trong nhóm, viết lại gọn trên cùng dữ liệu quý 3/2026 của Bước 1, gồm 4 bước:

1. **Chỉ tính shop đủ dữ liệu:** shop đang bán, không phải shop mới, có từ 30 hồ sơ trong quý (với tỷ lệ ký) hoặc từ 30 lead trong quý (với cross-sell). Peer là các shop đủ dữ liệu cùng peer group; nếu peer group có dưới 15 shop khác thì dùng shop cùng loại trên toàn quốc (cùng cỡ, ngành hàng, kiểu làm hồ sơ); nếu vẫn dưới 15 thì bỏ cỡ shop.
2. **Làm mượt tỷ lệ:** tỷ lệ của shop ít hồ sơ dao động mạnh do ngẫu nhiên, nên được kéo về gần mức chung của peer: `tỷ lệ làm mượt = (số thành công + 30 × mức chung của peer) / (số lần thử + 30)`. Cách này giống như cộng thêm 30 hồ sơ có tỷ lệ bằng mức chung (gọi là credibility weighting). Shop càng nhiều hồ sơ thì tỷ lệ làm mượt càng gần tỷ lệ thật của shop. Mức chung của peer không tính chính shop đó.
3. **So với một shop có thật:** trong các peer, chọn shop có tỷ lệ làm mượt gần mức 75% của peer nhất (shop donor). Benchmark là mức thấp hơn giữa tỷ lệ thật và tỷ lệ làm mượt của shop donor.
4. **Phần tăng** = số hồ sơ (hoặc lead) mỗi tháng × (benchmark − mức cao hơn giữa tỷ lệ thật và tỷ lệ làm mượt của shop), không âm. Lấy mức cao hơn của hai tỷ lệ để không coi một shop là kém chỉ vì ngẫu nhiên. Shop có từ 30 contract tính rủi ro và tỷ lệ trễ hạn cao hơn mức chung của peer group (mục 3.4) được giữ phần tăng bằng 0.

Câu lệnh dưới đây làm 4 bước trên cho hai nhánh (tỷ lệ ký, cross-sell mỗi lead), rồi in số shop đủ dữ liệu, số shop có phần tăng dương và tổng phần tăng mỗi tháng.

In [13]:
K_MUOT, MIN_LAN_THU, MIN_PEER = 30, 30, 15
CAP = [G, G_QG, ['salesroom_product', 'salesroom_model']]   # peer group, rồi toàn quốc cùng loại, rồi bỏ cỡ shop

def kiem_chung(e):
    # e: shop đủ dữ liệu, cột n = số lần thử trong quý, k = số thành công trong quý
    e = e.reset_index(drop=True).copy()
    e['r_tho'] = e.k / e.n
    e['cap_peer'] = -1
    for i, cap in enumerate(CAP):
        so_peer = e.groupby(cap).n.transform('size') - 1
        e.loc[(e.cap_peer < 0) & (so_peer >= MIN_PEER), 'cap_peer'] = i
    e = e[e.cap_peer >= 0].copy()
    e['p_peer'] = np.nan
    for i, cap in enumerate(CAP):
        sk, sn = e.groupby(cap).k.transform('sum'), e.groupby(cap).n.transform('sum')
        e.loc[e.cap_peer == i, 'p_peer'] = ((sk - e.k) / (sn - e.n))[e.cap_peer == i]
    e['r_muot'] = (e.k + K_MUOT * e.p_peer) / (e.n + K_MUOT)
    ket = []
    for _, r in e.iterrows():
        cap = CAP[int(r.cap_peer)]
        peer = e[(e[cap] == r[cap]).all(axis=1) & (e.skp_salesroom != r.skp_salesroom)]
        muc75 = peer.r_muot.quantile(0.75)
        dn = peer.loc[(peer.r_muot - muc75).abs().idxmin()]
        ket.append((dn.skp_salesroom, min(dn.r_tho, dn.r_muot)))
    e['donor'], e['bm'] = zip(*ket)
    e['tang'] = e.n / 3 * np.clip(e.bm - np.maximum(e.r_tho, e.r_muot), 0, None)
    return e[['skp_salesroom', 'n', 'k', 'r_tho', 'p_peer', 'r_muot', 'cap_peer', 'donor', 'bm', 'tang']]

nen = x[x.dang_ban & (x.shop_moi == 0)]
ky = kiem_chung(nen[nen.ca_ho_so * 3 >= MIN_LAN_THU].assign(n=lambda t: t.ca_ho_so * 3, k=lambda t: t.ca_hd_truc_tiep * 3))
xs = kiem_chung(nen[nen.ca_lead * 3 >= MIN_LAN_THU].assign(n=lambda t: t.ca_lead * 3, k=lambda t: t.ca_xsell * 3))
kc = x[['skp_salesroom', 'rr_hd', 'rr_ty_le_tre', 'rr_peer_group']].merge(
        ky.add_suffix('_ky').rename(columns={'skp_salesroom_ky': 'skp_salesroom'}), on='skp_salesroom', how='left').merge(
        xs.add_suffix('_xs').rename(columns={'skp_salesroom_xs': 'skp_salesroom'}), on='skp_salesroom', how='left')
kc = kc[kc.tang_ky.notna() | kc.tang_xs.notna()].copy()
kc['rui_ro'] = np.select([kc.rr_hd < 30, kc.rr_ty_le_tre > kc.rr_peer_group], ['chưa đủ dữ liệu rủi ro', 'giữ lại vì rủi ro'], 'không cao hơn peer')
kc['tang_truoc_rui_ro'] = kc.tang_ky.fillna(0) + kc.tang_xs.fillna(0)
kc['tang_kiem_chung'] = np.where(kc.rui_ro == 'giữ lại vì rủi ro', 0, kc.tang_truoc_rui_ro)
pd.DataFrame([{'nhanh': 'tỷ lệ ký', 'so_shop_du_du_lieu': len(ky), 'so_shop_tang_duong': int((ky.tang > 1e-9).sum()), 'tang_moi_thang': ky.tang.sum()},
              {'nhanh': 'cross-sell mỗi lead', 'so_shop_du_du_lieu': len(xs), 'so_shop_tang_duong': int((xs.tang > 1e-9).sum()), 'tang_moi_thang': xs.tang.sum()},
              {'nhanh': 'cả hai, trước khi giữ lại vì rủi ro', 'so_shop_du_du_lieu': len(kc), 'so_shop_tang_duong': int((kc.tang_truoc_rui_ro > 1e-9).sum()), 'tang_moi_thang': kc.tang_truoc_rui_ro.sum()},
              {'nhanh': 'cả hai, sau khi giữ lại vì rủi ro', 'so_shop_du_du_lieu': len(kc), 'so_shop_tang_duong': int((kc.tang_kiem_chung > 1e-9).sum()), 'tang_moi_thang': kc.tang_kiem_chung.sum()}]).set_index('nhanh')

,so_shop_du_du_lieu,so_shop_tang_duong,tang_moi_thang
nhanh,,,
tỷ lệ ký,1041,654,"1,599.9"
cross-sell mỗi lead,1004,708,493.2
"cả hai, trước khi giữ lại vì rủi ro",1342,1099,"2,093.1"
"cả hai, sau khi giữ lại vì rủi ro",1342,744,"1,355.7"


- Với tỷ lệ ký, 1.041 shop đủ dữ liệu; 654 shop có tỷ lệ thấp hơn benchmark, phần tăng 1.599,9 contract mỗi tháng. Với cross-sell, 1.004 shop đủ dữ liệu, phần tăng 493,2.
- Gộp hai nhánh có 1.342 shop. Sau khi giữ phần tăng bằng 0 ở các shop có tỷ lệ trễ hạn cao hơn peer, còn 744 shop có phần tăng dương, tổng 1.355,7 contract mỗi tháng. Bước giữ lại vì rủi ro bỏ đi 35% phần tăng (từ 2.093,1 xuống 1.355,7).

Câu lệnh dưới đây in số shop theo nhãn rủi ro, phần trăm shop đủ dữ liệu đã ký nhiều hơn mức 75% số contract của peer group và phần trăm chạm giới hạn ở mục 3.5, rồi so trên đúng các shop đủ dữ liệu này hai con số:
- phần tăng theo cách kiểm chứng (cột `tang_kiem_chung`);
- phần tăng nhờ tỷ lệ của Attainable Potential ở mục 3.7 (tỷ lệ bị từ chối + tỷ lệ hủy + cross-sell), và tổng phần Attainable Potential cao hơn BAU của cùng các shop.

In [14]:
display(kc.rui_ro.value_counts().to_frame('so_shop'))
m_ = x[x.skp_salesroom.isin(kc.skp_salesroom)]
print(f"Trong {len(m_):,} shop đủ dữ liệu: {(m_.ca_hd_truc_tiep >= m_.bm_hd_truc_tiep).mean():.1%} đang ký nhiều hơn mức 75% số contract của peer group; "
      f"{(m_.tt_tinh_lai > m_.tran_nhom + 1e-9).mean():.1%} chạm giới hạn ở mục 3.5")
so = kc[['skp_salesroom', 'tang_kiem_chung']].merge(
    x[['skp_salesroom', 'ap_hd_tong', 'bau_hd_tong', 'tang_do_ty_le_tu_choi', 'tang_do_ty_le_huy', 'tang_do_xsell']], on='skp_salesroom')
pd.DataFrame({'so_shop': [len(so)],
              'kiem_chung_chi_nang_ty_le': [so.tang_kiem_chung.sum()],
              'muc_3_7_phan_nho_ty_le': [so[['tang_do_ty_le_tu_choi', 'tang_do_ty_le_huy', 'tang_do_xsell']].sum().sum()],
              'muc_3_7_tong_ap_tru_bau': [(so.ap_hd_tong - so.bau_hd_tong).sum()],
              'ty_trong_trong_ap_tru_bau_toan_mang': [(so.ap_hd_tong - so.bau_hd_tong).sum() / (x.ap_hd_tong - x.bau_hd_tong).sum()]})

,so_shop
rui_ro,
không cao hơn peer,553
giữ lại vì rủi ro,425
chưa đủ dữ liệu rủi ro,364


Trong 1,342 shop đủ dữ liệu: 86.8% đang ký nhiều hơn mức 75% số contract của peer group; 90.3% chạm giới hạn ở mục 3.5


,so_shop,kiem_chung_chi_nang_ty_le,muc_3_7_phan_nho_ty_le,muc_3_7_tong_ap_tru_bau,ty_trong_trong_ap_tru_bau_toan_mang
0,1342,"1,355.7",613.8,"2,680.4",0.130


- 425 trên 1.342 shop bị giữ lại vì rủi ro. 364 shop chưa đủ 30 contract để đo rủi ro; các shop này vẫn được tính phần tăng nhưng cần xem lại rủi ro trước khi làm.
- Trên cùng 1.342 shop, cách kiểm chứng cho phần tăng nhờ tỷ lệ là 1.355,7 contract mỗi tháng, gấp khoảng 2,2 lần phần tỷ lệ của mục 3.7 (613,8). Lý do: 86,8% các shop này đang ký nhiều hơn mức 75% số contract của peer group và 90,3% chạm giới hạn ở mục 3.5. Phần bị giới hạn cắt đi được chia theo logarit của từng KPI, nên phần của các tỷ lệ bị cắt nhiều. Vì vậy phần tăng nhờ tỷ lệ trong Attainable Potential là con số thận trọng.
- Các shop đủ dữ liệu chỉ giữ 13% phần Attainable Potential cao hơn BAU của toàn mạng lưới (2.680,4 trên 20.589,5 contract mỗi tháng). 87% còn lại nằm ở shop nhỏ và shop mới, nơi không đủ hồ sơ để đo tỷ lệ của riêng shop; với các shop này chỉ dùng được cách so với peer group ở các mục trước.

**Kiểm tra theo thời gian: làm mượt có giúp dự báo tỷ lệ ký tốt hơn không.** Câu lệnh dưới đây lấy các shop có từ 30 hồ sơ ở cả quý 2/2026 (04–06) và quý 3/2026, dùng tỷ lệ ký của quý 2 để đoán tỷ lệ ký của quý 3 theo ba cách: tỷ lệ thật của quý 2, tỷ lệ làm mượt của quý 2 (kéo về mức chung của shop cùng loại trên toàn quốc, như bước 2 ở trên), và chỉ dùng mức chung đó. Sai số là chênh lệch tuyệt đối trung bình giữa tỷ lệ đoán và tỷ lệ thật của quý 3, tính theo trọng số số hồ sơ quý 3 (weighted MAE). Cách nào có sai số nhỏ hơn thì phản ánh năng lực thật của shop tốt hơn.

In [15]:
sm_ = pd.read_csv(ROOT / 'Bước 1' / 'shop_month.csv', encoding='utf-8-sig')
def quy(tu, den):
    return sm_[(sm_.thang >= tu) & (sm_.thang <= den)].groupby('skp_salesroom')[['ho_so', 'hd_ky']].sum()
t2 = d[['skp_salesroom'] + G_QG].join(quy('2026-04-01', '2026-06-01'), on='skp_salesroom').join(
        quy('2026-07-01', '2026-09-01').add_suffix('_q3'), on='skp_salesroom')
t2 = t2[(t2.ho_so >= MIN_LAN_THU) & (t2.ho_so_q3 >= MIN_LAN_THU)].copy()
sk, sn = t2.groupby(G_QG).hd_ky.transform('sum'), t2.groupby(G_QG).ho_so.transform('sum')
t2['chung'] = (sk - t2.hd_ky) / (sn - t2.ho_so)
t2['tho'] = t2.hd_ky / t2.ho_so
t2['muot'] = (t2.hd_ky + K_MUOT * t2.chung) / (t2.ho_so + K_MUOT)
t2['that_q3'] = t2.hd_ky_q3 / t2.ho_so_q3
mae = lambda c: np.average((t2[c] - t2.that_q3).abs(), weights=t2.ho_so_q3)
pd.DataFrame({'so_shop': len(t2), 'sai_so': [mae('tho'), mae('muot'), mae('chung')]},
             index=['tỷ lệ thật quý 2', 'tỷ lệ làm mượt quý 2', 'chỉ mức chung của shop cùng loại'])

,so_shop,sai_so
tỷ lệ thật quý 2,815,0.080
tỷ lệ làm mượt quý 2,815,0.073
chỉ mức chung của shop cùng loại,815,0.094


- Trên 815 shop, tỷ lệ làm mượt đoán tỷ lệ ký quý 3 sát nhất: sai số 0,073, so với 0,080 khi dùng tỷ lệ thật của quý 2 và 0,094 khi chỉ dùng mức chung. Nghĩa là tỷ lệ của một quý có một phần là ngẫu nhiên, và kéo nó về mức chung giúp phản ánh năng lực của shop tốt hơn.
- Mức cải thiện khoảng 9%, trên các shop có từ 30 hồ sơ mỗi quý. Shop ít hồ sơ hơn có tỷ lệ dựa trên ít hồ sơ hơn nên phần ngẫu nhiên lớn hơn; đây cũng là lý do mục 3.2 chỉ tính benchmark tỷ lệ trên shop có từ 10 hồ sơ.

Ô dưới đây ghi kết quả kiểm chứng của từng shop đủ dữ liệu vào `Bước 3/shop_kiem_chung_ty_le.csv`: số lần thử và số thành công trong quý, tỷ lệ thật, tỷ lệ làm mượt, shop donor, benchmark và phần tăng của từng nhánh (hậu tố `_ky` là tỷ lệ ký, `_xs` là cross-sell mỗi lead), nhãn rủi ro, phần tăng trước và sau khi giữ lại vì rủi ro.

In [16]:
kc.sort_values('skp_salesroom').to_csv(ROOT / 'Bước 3' / 'shop_kiem_chung_ty_le.csv', index=False, encoding='utf-8-sig')
print(f'Đã ghi Bước 3/shop_kiem_chung_ty_le.csv: {len(kc):,} dòng')

Đã ghi Bước 3/shop_kiem_chung_ty_le.csv: 1,342 dòng


## 3.11. Tóm tắt

- Peer group gồm cỡ shop × ngành hàng × kiểu làm hồ sơ × tỉnh mới. Benchmark là mức mà 1/4 số shop cùng nhóm đã đạt được. Nhóm dưới 5 shop dùng benchmark toàn quốc.
- Hai giới hạn giữ con số thực tế: chỉ giảm tỷ lệ bị từ chối khi shop có đủ 30 contract và khách không trễ hạn nhiều hơn mức chung của nhóm; và kết quả không vượt số contract mà 1/4 shop tốt nhất của nhóm đang ký.
- Tổng Attainable Potential của các shop được đánh giá là khoảng 59.900 contract mỗi tháng, cao hơn BAU khoảng 52%. Shop TW có khả năng tăng thêm lớn nhất so với BAU (62–71%).
- Phần tăng thêm chủ yếu nằm ở số hồ sơ shop tạo ra: hồ sơ mỗi SA (42,7%) và số SA (32,8%). Tỷ lệ hủy (1,6%) và tỷ lệ bị từ chối (0,4%) đóng góp ít; tỷ lệ bị từ chối còn bị giới hạn bởi rủi ro.
- Kiểm chứng chỉ nâng tỷ lệ, giữ nguyên số hồ sơ (mục 3.10, theo cách làm của Nguyên): trên 1.342 shop đủ dữ liệu, phần tăng nhờ tỷ lệ là 1.355,7 contract mỗi tháng, gấp 2,2 lần phần tỷ lệ trong Attainable Potential của cùng các shop (613,8). Phần tỷ lệ trong Attainable Potential vì vậy là con số thận trọng. Làm mượt tỷ lệ giảm sai số dự báo tỷ lệ ký từ 0,080 xuống 0,073.
- Kết quả chưa áp trần thị trường. Bước 4 tính Addressable Market Ceiling; Bước 5 giới hạn Attainable Potential không vượt trần đó.
- Cách đọc kết quả: với mỗi shop, cột `ap_hd_tong` trong `output/shop_attainable.csv` bằng N nghĩa là nếu shop làm tốt bằng 1/4 số shop cùng loại đang làm, shop sẽ ký được N contract mỗi tháng.